In [1]:
import pandas as pd
from pathlib import Path

class_dirs = {
    "low_density": Path("/kaggle/input/datasets/projectburak/lowdensity1"),
    "medium_density": Path("/kaggle/input/datasets/projectburak/medium-density"),
    "high_density": Path("/kaggle/input/datasets/projectburak/highdensity1"),
}

CLASS_NAMES = ["low_density", "medium_density", "high_density"]
class_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}

valid_ext = [".jpg", ".jpeg", ".png", ".bmp", ".webp", ".jfif"]

data = []

for class_name, folder in class_dirs.items():
    for p in folder.rglob("*"):
        if p.is_file() and p.suffix.lower() in valid_ext:
            data.append({
                "path": str(p),
                "label": class_to_idx[class_name],
                "class_name": class_name
            })

df = pd.DataFrame(data)

print(df["class_name"].value_counts())
print("Toplam:", len(df))
print("Class mapping:", class_to_idx)
df.head()

class_name
high_density      3270
low_density       1632
medium_density    1125
Name: count, dtype: int64
Toplam: 6027
Class mapping: {'low_density': 0, 'medium_density': 1, 'high_density': 2}


,path,label,class_name
0,/kaggle/input/datasets/projectburak/lowdensity...,0,low_density
1,/kaggle/input/datasets/projectburak/lowdensity...,0,low_density
2,/kaggle/input/datasets/projectburak/lowdensity...,0,low_density
3,/kaggle/input/datasets/projectburak/lowdensity...,0,low_density
4,/kaggle/input/datasets/projectburak/lowdensity...,0,low_density


In [2]:
from sklearn.model_selection import train_test_split

SEED = 42

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

print("Train:", len(train_df))
print(train_df["class_name"].value_counts())

print("\nValidation:", len(val_df))
print(val_df["class_name"].value_counts())

print("\nTest:", len(test_df))
print(test_df["class_name"].value_counts())

Train: 4218
class_name
high_density      2289
low_density       1142
medium_density     787
Name: count, dtype: int64

Validation: 904
class_name
high_density      490
low_density       245
medium_density    169
Name: count, dtype: int64

Test: 905
class_name
high_density      491
low_density       245
medium_density    169
Name: count, dtype: int64


In [3]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMG_SIZE = 260
BATCH_SIZE = 32
NUM_CLASSES = 3
CLASS_NAMES = ["low_density", "medium_density", "high_density"]

print("Device:", DEVICE)
print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Classes:", CLASS_NAMES)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
Image size: 260
Batch size: 32
Classes: ['low_density', 'medium_density', 'high_density']
GPU: Tesla T4


In [4]:
from torchvision import transforms

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15,
        hue=0.03
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transformlar hazır.")

Transformlar hazır.


In [5]:
from PIL import Image
from torch.utils.data import Dataset

Image.MAX_IMAGE_PIXELS = None

class DensityDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, "path"]
        label = int(self.df.loc[idx, "label"])

        with Image.open(img_path) as img:
            image = img.convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

print("DensityDataset hazır.")

DensityDataset hazır.


In [6]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    DensityDataset(train_df, train_tfms),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    DensityDataset(val_df, eval_tfms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    DensityDataset(test_df, eval_tfms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train batch sayısı:", len(train_loader))
print("Validation batch sayısı:", len(val_loader))
print("Test batch sayısı:", len(test_loader))

Train batch sayısı: 132
Validation batch sayısı: 29
Test batch sayısı: 29


In [7]:
import torch
import torch.nn as nn

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction
        self.ce = nn.CrossEntropyLoss(reduction="none")

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.alpha is not None:
            alpha_t = self.alpha.to(inputs.device)[targets]
            focal_loss = alpha_t * focal_loss

        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        else:
            return focal_loss

criterion = FocalLoss(gamma=2.0)

print("Focal Loss hazır.")

Focal Loss hazır.


In [8]:
!pip install -q timm

import timm

model = timm.create_model(
    "efficientnet_b2",
    pretrained=True,
    num_classes=NUM_CLASSES
)

model = model.to(DEVICE)

print("Model hazır: EfficientNet-B2 Density Module")
print("Device:", next(model.parameters()).device)
print("Class count:", NUM_CLASSES)

model.safetensors:   0%|          | 0.00/36.8M [00:00<?, ?B/s]

Model hazır: EfficientNet-B2 Density Module
Device: cuda:0
Class count: 3


In [9]:
from sklearn.metrics import accuracy_score, f1_score

def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None

    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, macro_f1

print("Epoch fonksiyonu hazır.")

Epoch fonksiyonu hazır.


In [10]:
import time
import torch.optim as optim

BEST_MODEL_PATH = "/kaggle/working/b2_density_best_model.pth"
history = []
best_macro_f1_global = -1.0

def train_stage(stage_name, start_epoch, end_epoch, lr, freeze_backbone=False):
    global best_macro_f1_global, history

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

        classifier = model.get_classifier()
        for param in classifier.parameters():
            param.requires_grad = True
    else:
        for param in model.parameters():
            param.requires_grad = True

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=1e-4
    )

    for epoch in range(start_epoch, end_epoch + 1):
        start_time = time.time()

        train_loss, train_acc, train_macro_f1 = run_one_epoch(
            model, train_loader, criterion, optimizer
        )

        val_loss, val_acc, val_macro_f1 = run_one_epoch(
            model, val_loader, criterion, optimizer=None
        )

        elapsed = time.time() - start_time

        history.append({
            "stage": stage_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "val_macro_f1": val_macro_f1,
            "time_sec": elapsed
        })

        print(
            f"{stage_name} | Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.4f} | "
            f"val_macro_f1={val_macro_f1:.4f}"
        )

        if val_macro_f1 > best_macro_f1_global:
            best_macro_f1_global = val_macro_f1
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "val_macro_f1": val_macro_f1,
                "val_accuracy": val_acc,
                "class_names": CLASS_NAMES,
                "img_size": IMG_SIZE
            }, BEST_MODEL_PATH)
            print("Best model saved:", BEST_MODEL_PATH)

print("Eğitim fonksiyonu hazır.")

Eğitim fonksiyonu hazır.


In [11]:
train_stage(
    stage_name="stage1",
    start_epoch=1,
    end_epoch=10,
    lr=1e-3,
    freeze_backbone=True
)

stage1 | Epoch 01 | train_loss=1.6038 | val_loss=1.0057 | val_acc=0.6659 | val_macro_f1=0.6083
Best model saved: /kaggle/working/b2_density_best_model.pth
stage1 | Epoch 02 | train_loss=0.9514 | val_loss=0.7820 | val_acc=0.7257 | val_macro_f1=0.6695
Best model saved: /kaggle/working/b2_density_best_model.pth
stage1 | Epoch 03 | train_loss=0.7678 | val_loss=0.6465 | val_acc=0.7688 | val_macro_f1=0.7129
Best model saved: /kaggle/working/b2_density_best_model.pth
stage1 | Epoch 04 | train_loss=0.6461 | val_loss=0.5880 | val_acc=0.7644 | val_macro_f1=0.7116
stage1 | Epoch 05 | train_loss=0.6496 | val_loss=0.5470 | val_acc=0.7987 | val_macro_f1=0.7535
Best model saved: /kaggle/working/b2_density_best_model.pth
stage1 | Epoch 06 | train_loss=0.5190 | val_loss=0.4866 | val_acc=0.8197 | val_macro_f1=0.7708
Best model saved: /kaggle/working/b2_density_best_model.pth
stage1 | Epoch 07 | train_loss=0.5182 | val_loss=0.4574 | val_acc=0.8230 | val_macro_f1=0.7807
Best model saved: /kaggle/working/b

In [12]:
train_stage(
    stage_name="stage2",
    start_epoch=11,
    end_epoch=30,
    lr=1e-4,
    freeze_backbone=False
)

stage2 | Epoch 11 | train_loss=0.3693 | val_loss=0.2578 | val_acc=0.8905 | val_macro_f1=0.8603
Best model saved: /kaggle/working/b2_density_best_model.pth
stage2 | Epoch 12 | train_loss=0.1592 | val_loss=0.1889 | val_acc=0.9038 | val_macro_f1=0.8764
Best model saved: /kaggle/working/b2_density_best_model.pth
stage2 | Epoch 13 | train_loss=0.1216 | val_loss=0.1639 | val_acc=0.8993 | val_macro_f1=0.8706
stage2 | Epoch 14 | train_loss=0.0809 | val_loss=0.1359 | val_acc=0.9038 | val_macro_f1=0.8819
Best model saved: /kaggle/working/b2_density_best_model.pth
stage2 | Epoch 15 | train_loss=0.0513 | val_loss=0.1331 | val_acc=0.9060 | val_macro_f1=0.8776
stage2 | Epoch 16 | train_loss=0.0410 | val_loss=0.1231 | val_acc=0.9082 | val_macro_f1=0.8848
Best model saved: /kaggle/working/b2_density_best_model.pth
stage2 | Epoch 17 | train_loss=0.0316 | val_loss=0.1068 | val_acc=0.9248 | val_macro_f1=0.9048
Best model saved: /kaggle/working/b2_density_best_model.pth
stage2 | Epoch 18 | train_loss=0.03

In [13]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)

print("Devam için yüklenen epoch:", checkpoint["epoch"])
print("Mevcut en iyi val_macro_f1:", checkpoint["val_macro_f1"])

train_stage(
    stage_name="stage3",
    start_epoch=31,
    end_epoch=40,
    lr=2e-5,
    freeze_backbone=False
)

Devam için yüklenen epoch: 23
Mevcut en iyi val_macro_f1: 0.9069276082516243
stage3 | Epoch 31 | train_loss=0.0150 | val_loss=0.1247 | val_acc=0.9181 | val_macro_f1=0.8974
stage3 | Epoch 32 | train_loss=0.0123 | val_loss=0.1229 | val_acc=0.9170 | val_macro_f1=0.8947
stage3 | Epoch 33 | train_loss=0.0080 | val_loss=0.1357 | val_acc=0.9192 | val_macro_f1=0.8949
stage3 | Epoch 34 | train_loss=0.0073 | val_loss=0.1213 | val_acc=0.9248 | val_macro_f1=0.9047
stage3 | Epoch 35 | train_loss=0.0095 | val_loss=0.1338 | val_acc=0.9325 | val_macro_f1=0.9126
Best model saved: /kaggle/working/b2_density_best_model.pth
stage3 | Epoch 36 | train_loss=0.0103 | val_loss=0.1387 | val_acc=0.9248 | val_macro_f1=0.9007
stage3 | Epoch 37 | train_loss=0.0053 | val_loss=0.1439 | val_acc=0.9292 | val_macro_f1=0.9081
stage3 | Epoch 38 | train_loss=0.0074 | val_loss=0.1238 | val_acc=0.9248 | val_macro_f1=0.9005
stage3 | Epoch 39 | train_loss=0.0038 | val_loss=0.1282 | val_acc=0.9259 | val_macro_f1=0.9036
stage3 |

In [14]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import numpy as np
import torch

checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

print("Loaded checkpoint epoch:", checkpoint["epoch"])
print("Best validation macro F1:", checkpoint["val_macro_f1"])
print("Best validation accuracy:", checkpoint["val_accuracy"])

all_preds = []
all_labels = []
total_loss = 0.0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

test_loss = total_loss / len(test_loader.dataset)
test_acc = accuracy_score(all_labels, all_preds)
test_macro_f1 = f1_score(all_labels, all_preds, average="macro")

report_dict = classification_report(
    all_labels,
    all_preds,
    target_names=CLASS_NAMES,
    digits=4,
    output_dict=True
)

report_text = classification_report(
    all_labels,
    all_preds,
    target_names=CLASS_NAMES,
    digits=4
)

cm = confusion_matrix(all_labels, all_preds)

print("\nTEST RESULTS")
print("Test Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_acc, 4))
print("Test Macro F1:", round(test_macro_f1, 4))

print("\nClassification Report:")
print(report_text)

print("\nConfusion Matrix:")
print(cm)

Loaded checkpoint epoch: 35
Best validation macro F1: 0.912617536943074
Best validation accuracy: 0.9325221238938053

TEST RESULTS
Test Loss: 0.0927
Test Accuracy: 0.9304
Test Macro F1: 0.9026

Classification Report:
                precision    recall  f1-score   support

   low_density     0.9066    0.9510    0.9283       245
medium_density     0.8600    0.7633    0.8088       169
  high_density     0.9639    0.9776    0.9707       491

      accuracy                         0.9304       905
     macro avg     0.9102    0.8973    0.9026       905
  weighted avg     0.9290    0.9304    0.9290       905


Confusion Matrix:
[[233  11   1]
 [ 23 129  17]
 [  1  10 480]]


In [15]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import numpy as np
import torch

checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

print("Loaded checkpoint epoch:", checkpoint["epoch"])
print("Best validation macro F1:", checkpoint["val_macro_f1"])
print("Best validation accuracy:", checkpoint["val_accuracy"])

all_preds = []
all_labels = []
total_loss = 0.0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

test_loss = total_loss / len(test_loader.dataset)
test_acc = accuracy_score(all_labels, all_preds)
test_macro_f1 = f1_score(all_labels, all_preds, average="macro")

report_dict = classification_report(
    all_labels,
    all_preds,
    target_names=CLASS_NAMES,
    digits=4,
    output_dict=True
)

report_text = classification_report(
    all_labels,
    all_preds,
    target_names=CLASS_NAMES,
    digits=4
)

cm = confusion_matrix(all_labels, all_preds)

print("\nTEST RESULTS")
print("Test Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_acc, 4))
print("Test Macro F1:", round(test_macro_f1, 4))

print("\nClassification Report:")
print(report_text)

print("\nConfusion Matrix:")
print(cm)

Loaded checkpoint epoch: 35
Best validation macro F1: 0.912617536943074
Best validation accuracy: 0.9325221238938053

TEST RESULTS
Test Loss: 0.0927
Test Accuracy: 0.9304
Test Macro F1: 0.9026

Classification Report:
                precision    recall  f1-score   support

   low_density     0.9066    0.9510    0.9283       245
medium_density     0.8600    0.7633    0.8088       169
  high_density     0.9639    0.9776    0.9707       491

      accuracy                         0.9304       905
     macro avg     0.9102    0.8973    0.9026       905
  weighted avg     0.9290    0.9304    0.9290       905


Confusion Matrix:
[[233  11   1]
 [ 23 129  17]
 [  1  10 480]]


In [16]:
!pip install -q reportlab seaborn

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from reportlab.platypus import SimpleDocTemplate, Paragraph, Image as RLImage, Table, TableStyle, PageBreak
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch

REPORT_PDF = "/kaggle/working/efficientnet_b2_density_report.pdf"

history_df = pd.DataFrame(history)
history_df.to_csv("/kaggle/working/b2_density_epoch_results.csv", index=False)

# Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.title("Test Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
cm_path = "/kaggle/working/b2_density_confusion_matrix.png"
plt.savefig(cm_path, dpi=300)
plt.close()

# Validation Macro F1
plt.figure(figsize=(9, 5))
plt.plot(history_df["epoch"], history_df["val_macro_f1"], marker="o")
plt.title("EfficientNet-B2 Validation Macro F1 Curve")
plt.xlabel("Epoch")
plt.ylabel("Validation Macro F1-score")
plt.grid(True)
plt.tight_layout()
f1_path = "/kaggle/working/b2_density_val_macro_f1_curve.png"
plt.savefig(f1_path, dpi=300)
plt.close()

# Train Loss vs Validation Loss
plt.figure(figsize=(9, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train Loss")
plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation Loss")
plt.title("EfficientNet-B2 Train / Validation Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
loss_path = "/kaggle/working/b2_density_loss_curve.png"
plt.savefig(loss_path, dpi=300)
plt.close()

styles = getSampleStyleSheet()
doc = SimpleDocTemplate(REPORT_PDF, pagesize=A4)
story = []

best_row = history_df.loc[history_df["val_macro_f1"].idxmax()]

summary_text = f"""
EfficientNet-B2 Forest Density Classification Report<br/>
<br/>
Model: EfficientNet-B2<br/>
Image Size: {IMG_SIZE}x{IMG_SIZE}<br/>
Batch Size: {BATCH_SIZE}<br/>
Optimizer: AdamW<br/>
Loss: Focal Loss (gamma=2.0)<br/>
Checkpoint Path: {BEST_MODEL_PATH}<br/>
<br/>
Dataset Split:<br/>
Train size: {len(train_df)}<br/>
Validation size: {len(val_df)}<br/>
Test size: {len(test_df)}<br/>
Total size: {len(df)}<br/>
<br/>
Class Mapping:<br/>
low_density = 0<br/>
medium_density = 1<br/>
high_density = 2<br/>
<br/>
Loaded Checkpoint:<br/>
Checkpoint epoch: {checkpoint["epoch"]}<br/>
Best Validation Macro F1: {checkpoint["val_macro_f1"]:.4f}<br/>
Best Validation Accuracy: {checkpoint["val_accuracy"]:.4f}<br/>
<br/>
TEST RESULTS:<br/>
Test Loss: {test_loss:.4f}<br/>
Test Accuracy: {test_acc:.4f}<br/>
Test Macro F1: {test_macro_f1:.4f}<br/>
<br/>
Low-Density Class:<br/>
Precision: {report_dict["low_density"]["precision"]:.4f}<br/>
Recall: {report_dict["low_density"]["recall"]:.4f}<br/>
F1-score: {report_dict["low_density"]["f1-score"]:.4f}<br/>
<br/>
Medium-Density Class:<br/>
Precision: {report_dict["medium_density"]["precision"]:.4f}<br/>
Recall: {report_dict["medium_density"]["recall"]:.4f}<br/>
F1-score: {report_dict["medium_density"]["f1-score"]:.4f}<br/>
<br/>
High-Density Class:<br/>
Precision: {report_dict["high_density"]["precision"]:.4f}<br/>
Recall: {report_dict["high_density"]["recall"]:.4f}<br/>
F1-score: {report_dict["high_density"]["f1-score"]:.4f}<br/>
"""

story.append(Paragraph(summary_text, styles["Normal"]))
story.append(PageBreak())

story.append(Paragraph("Classification Report - Test Set", styles["Title"]))
report_table_data = [["class", "precision", "recall", "f1-score", "support"]]

for cls in CLASS_NAMES:
    report_table_data.append([
        cls,
        f'{report_dict[cls]["precision"]:.4f}',
        f'{report_dict[cls]["recall"]:.4f}',
        f'{report_dict[cls]["f1-score"]:.4f}',
        int(report_dict[cls]["support"])
    ])

report_table_data.append([
    "accuracy",
    "",
    "",
    f'{report_dict["accuracy"]:.4f}',
    len(test_df)
])

report_table_data.append([
    "macro avg",
    f'{report_dict["macro avg"]["precision"]:.4f}',
    f'{report_dict["macro avg"]["recall"]:.4f}',
    f'{report_dict["macro avg"]["f1-score"]:.4f}',
    int(report_dict["macro avg"]["support"])
])

report_table_data.append([
    "weighted avg",
    f'{report_dict["weighted avg"]["precision"]:.4f}',
    f'{report_dict["weighted avg"]["recall"]:.4f}',
    f'{report_dict["weighted avg"]["f1-score"]:.4f}',
    int(report_dict["weighted avg"]["support"])
])

table = Table(report_table_data)
table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.black),
    ("ALIGN", (1, 1), (-1, -1), "CENTER"),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
]))
story.append(table)
story.append(PageBreak())

story.append(Paragraph("Test Confusion Matrix", styles["Title"]))
story.append(RLImage(cm_path, width=5.8*inch, height=4.8*inch))
story.append(PageBreak())

training_text = f"""
Model Training Report<br/>
<br/>
Model: EfficientNet-B2<br/>
Image Size: {IMG_SIZE}x{IMG_SIZE}<br/>
Batch Size: {BATCH_SIZE}<br/>
Optimizer: AdamW<br/>
Loss: Focal Loss<br/>
Primary Metric: Validation Macro F1-score<br/>
<br/>
Best Validation Result:<br/>
Stage: {best_row["stage"]}<br/>
Epoch: {int(best_row["epoch"])}<br/>
Val Macro F1: {best_row["val_macro_f1"]:.4f}<br/>
Val Accuracy: {best_row["val_accuracy"]:.4f}<br/>
Val Loss: {best_row["val_loss"]:.4f}<br/>
<br/>
Best model path:<br/>
{BEST_MODEL_PATH}<br/>
"""

story.append(Paragraph(training_text, styles["Normal"]))
story.append(PageBreak())

story.append(Paragraph("Epoch Results", styles["Title"]))

epoch_cols = ["stage", "epoch", "train_loss", "val_loss", "val_accuracy", "val_macro_f1", "time_sec"]
epoch_table_data = [epoch_cols]

for _, row in history_df[epoch_cols].iterrows():
    epoch_table_data.append([
        row["stage"],
        int(row["epoch"]),
        f'{row["train_loss"]:.4f}',
        f'{row["val_loss"]:.4f}',
        f'{row["val_accuracy"]:.4f}',
        f'{row["val_macro_f1"]:.4f}',
        f'{row["time_sec"]:.4f}'
    ])

epoch_table = Table(epoch_table_data, repeatRows=1)
epoch_table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("GRID", (0, 0), (-1, -1), 0.35, colors.black),
    ("ALIGN", (1, 1), (-1, -1), "CENTER"),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("FONTSIZE", (0, 0), (-1, -1), 7),
]))
story.append(epoch_table)
story.append(PageBreak())

story.append(Paragraph("Validation Macro F1 Over Epochs", styles["Title"]))
story.append(RLImage(f1_path, width=6.8*inch, height=3.8*inch))
story.append(PageBreak())

story.append(Paragraph("Train Loss vs Validation Loss", styles["Title"]))
story.append(RLImage(loss_path, width=6.8*inch, height=3.8*inch))

doc.build(story)

print("PDF rapor oluşturuldu:")
print(REPORT_PDF)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.8 MB/s eta 0:00:00
PDF rapor oluşturuldu:
/kaggle/working/efficientnet_b2_density_report.pdf
